# 00 — Initial Data Quality Assessment & Master Cleaning Pipeline (Thisara)

**Owner:** Thisara (Data Engineering Lead)  
**Covers:** Preliminary Data Analysis — Initial Data Quality Assessment (15 marks) + Master Cleaned Dataset Freeze (`data/processed/cleaned_crime_data.parquet`).

### Pipeline Objectives
1. Load raw `dataset_coursework1.csv` safely without silent dtype coercion.
2. Audit **shape, schema, missing values (%), exact duplicates, and Incident vs. Offence granularity**.
3. Audit **datetime anomalies** (out-of-range dates, `End_Date_Time < Start_Date_Time`, missing `Dispatch Date / Time`).
4. Audit **spatial & categorical noise** (zero/out-of-bounds `Latitude`/`Longitude`, whitespace/casing in `City`/`State`/`Location`, `Zip Code` formats).
5. Compare cleaning & missing-data handling strategies in Python and quantify `% of records affected`.
6. Export the single frozen master dataset (`cleaned_crime_data.parquet`) for Sachithra, Thisara, and Kavindu.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path("..").resolve()))
from utils.viz_style import apply_style, save_fig, RAW_DATA_PATH, CLEAN_DATA_PATH

apply_style()
pd.set_option("display.max_columns", 50)
print("Raw CSV:", RAW_DATA_PATH)
print("Target Parquet:", CLEAN_DATA_PATH)


## 1. Load Raw Dataset & Initial Schema Audit


In [ ]:
# Load with low_memory=False to avoid mixed-type warnings
df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f"Rows: {len(df_raw):,} | Columns: {df_raw.shape[1]}")
display(df_raw.head(3))
df_raw.info()


## 2. Missing Values Audit (% of Records Affected)


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(dropna=True),
    "dtype": df_raw.dtypes.astype(str),
}).sort_values("missing_pct", ascending=False)

display(missing_summary)


## 3. Duplicates & NIBRS Granularity (Incident vs. Offence vs. Victim)


In [ ]:
exact_dupes = df_raw.duplicated().sum()
n_incidents = df_raw["Incident ID"].nunique()
n_rows = len(df_raw)
avg_offences_per_incident = n_rows / n_incidents

print(f"Exact duplicate rows: {exact_dupes:,} ({exact_dupes / n_rows * 100:.4f}%)")
print(f"Total offence rows:   {n_rows:,}")
print(f"Unique Incident IDs:  {n_incidents:,}")
print(f"Avg offences/incident:{avg_offences_per_incident:.3f}")
print("Victims summary:")
display(df_raw["Victims"].describe())


## 4. Datetime Parsing, Date-Range & Chronology Checks


In [ ]:
dt_cols = ["Dispatch Date / Time", "Start_Date_Time", "End_Date_Time"]
df = df_raw.copy()
for col in dt_cols:
    df[col] = pd.to_datetime(df[col], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

for col in dt_cols:
    print(f"{col:22s} -> min: {df[col].min()} | max: {df[col].max()} | nulls: {df[col].isna().sum():,}")

# Check records outside expected study window (2016-07-01 to 2026-09-10)
before_start = (df["Start_Date_Time"] < "2016-07-01").sum()
after_end = (df["Start_Date_Time"] > "2026-09-10 23:59:59").sum()
end_before_start = (df["End_Date_Time"] < df["Start_Date_Time"]).sum()

print(f"Start_Date_Time before 2016-07-01: {before_start:,} ({before_start/len(df)*100:.3f}%)")
print(f"Start_Date_Time after  2026-09-10: {after_end:,} ({after_end/len(df)*100:.3f}%)")
print(f"End_Date_Time < Start_Date_Time:   {end_before_start:,} ({end_before_start/len(df)*100:.3f}%)")


## 5. Spatial & Categorical Noise Audit


In [ ]:
# Montgomery County, MD approximate bounding box: Lat [38.9, 39.4], Lon [-77.6, -76.8]
zero_coords = ((df["Latitude"] == 0) | (df["Longitude"] == 0)).sum()
oob_coords = (
    df["Latitude"].notna() & df["Longitude"].notna() &
    ~((df["Latitude"].between(38.9, 39.4)) & (df["Longitude"].between(-77.6, -76.8)))
).sum()

print(f"Zero coordinates (0, 0):        {zero_coords:,} ({zero_coords/len(df)*100:.3f}%)")
print(f"Out-of-bounds coordinates:      {oob_coords:,} ({oob_coords/len(df)*100:.3f}%)")
print("\nTop 10 States:\n", df["State"].value_counts(dropna=False).head(10))
print("\nCrime Name1 distribution:\n", df["Crime Name1"].value_counts(dropna=False))
print("\nPolice District Name distribution:\n", df["Police District Name"].value_counts(dropna=False))


## 6. Cleaning, Missing-Data Method Comparison & Feature Engineering
*(Document your comparison of drop vs. flag/impute methods here, justify final decisions, and quantify % of records affected.)*


In [ ]:
# Example cleaning & feature engineering skeleton (refine after reviewing audit outputs above):
df_clean = df.drop_duplicates().copy()

# Strip whitespace on string columns
str_cols = df_clean.select_dtypes(include="object").columns
for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()

# Replace invalid (0, 0) coordinates with NaN instead of dropping entire crime records
invalid_coord_mask = (df_clean["Latitude"] == 0) | (df_clean["Longitude"] == 0)
df_clean.loc[invalid_coord_mask, ["Latitude", "Longitude"]] = np.nan

# Extract standard temporal features for downstream RQs
df_clean["Year"] = df_clean["Start_Date_Time"].dt.year
df_clean["Month"] = df_clean["Start_Date_Time"].dt.month
df_clean["DayOfWeek"] = df_clean["Start_Date_Time"].dt.day_name()
df_clean["Hour"] = df_clean["Start_Date_Time"].dt.hour

# Dispatch lag in hours (where Dispatch Date / Time is available)
df_clean["Dispatch_Lag_Hours"] = (
    (df_clean["Dispatch Date / Time"] - df_clean["Start_Date_Time"]).dt.total_seconds() / 3600.0
)

print(f"Cleaned dataset shape: {df_clean.shape}")


## 7. Freeze & Export Master Cleaned Dataset (Parquet)


In [ ]:
CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(CLEAN_DATA_PATH, index=False)
print(f"Frozen cleaned dataset saved -> {CLEAN_DATA_PATH} ({CLEAN_DATA_PATH.stat().st_size / 1e6:.2f} MB)")
